In [ ]:
import pandas as pd
import numpy as np
import re
import spacy
from transformers import pipeline

In [2]:
df = pd.read_excel('reviews_processed.xlsx')

In [3]:
nlp = spacy.load('ru_core_news_sm', disable=['ner', 'parser'])

In [4]:
sentiment_pipeline = pipeline('sentiment-analysis', model='seara/rubert-base-cased-russian-sentiment')

Loading weights: 100%|█████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 2972.27it/s]


In [5]:
def text_lemmat(text):
    if not isinstance(text, str):
        return ''

    clean_text = re.sub(r'[^\w\s]', ' ', text.lower())

    processed = nlp(clean_text) 

    return ' '.join([token.lemma_ for token in processed])

df['clean_text'] = df['Текст отзыва'].apply(text_lemmat)

In [6]:
df

,№,Сеть / формат,Адрес точки,Источник,Текст отзыва,Оценка,Дата,Город,clean_text
0,1,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,"Удобный магазин, вкусный кофе и выпечка. Быстр...",5.0,2025-10-18,Москва,удобный магазин вкусный кофе и выпечка быс...
1,2,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,"Доставили сюда посылку 5post, было указано что...",2.0,2026-04-27,Москва,доставить сюда посылка 5post было указать чт...
2,3,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,"Расположение удобное, продавцы улыбчивые, прия...",3.0,2025-11-27,Москва,расположение удобный продавец улыбчивый пр...
3,4,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,"Нормальный магазин. Не очень большой выбор, но...",4.0,2026-01-02,Москва,нормальный магазин не очень большой выбор ...
4,5,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,Кофе 3 в 1 80₽ и говорят такая цена. Через мес...,1.0,2025-11-30,Москва,кофе 3 в 1 80 и говорить такой цена через ...
...,...,...,...,...,...,...,...,...,...
252,253,Налету,NaN,Habr,"Кажется, сегодня ту Пятерочку настиг хабраэффе...",NaN,2021-02-24,NaN,казаться сегодня тот пятёрочка настигнуть ха...
253,254,Налету,NaN,Habr,"Где же мои глупые дети работать то будут, на п...",NaN,2021-03-01,NaN,где же мой глупый ребёнок работать то быть н...
254,255,Налету,NaN,Habr,"Ну, что заголовок кликбэйтный, это понятно :)\...",NaN,2021-03-08,NaN,ну что заголовок кликбэйтный это понятный ...
255,256,Налету,NaN,Habr,Пользуюсь сервисом в одном из Зеленоградских П...,NaN,2021-05-05,NaN,пользоваться сервис в одном из зеленоградский ...


In [7]:
themes = {
    'ассортимент': [
        'ассортимент', 'выбор', 'товар', 'полка', 'пусто', 'нет в наличии',
        'закончился', 'скудный', 'разнообразие', 'полки', 'найти', 'есть ли', 'дефицит'
    ],
    'цена': [
        'цена', 'дорого', 'дорогой', 'дешевый', 'стоимость', 'ценник',
        'акция', 'скидка', 'переплата', 'оверпрайс', 'переплатил',
        'бюджет', 'недорого', 'выгодно'
    ],
    'скорость': [
        'быстро', 'медленно', 'очередь', 'долго', 'ожидание',
        'время', 'касса', 'кассир', 'задержка', 'простоял',
        'ускорить', 'оперативно'
    ],
    'качество еды': [
        'вкусный', 'вкус', 'невкусный', 'свежий', 'несвежий',
        'просрочка', 'тухлый', 'кислый', 'пересолено',
        'салат', 'блюдо', 'еда', 'отравился', 'холодный', 'выпечка'
    ],
    'сервис': [
        'персонал', 'вежливый', 'хамство', 'грубый', 'помочь',
        'обслуживание', 'кассир', 'терминал', 'сотрудник',
        'поддержка', 'игнор', 'саппорт', 'обращение',
        'приложение', 'интерфейс', 'логин', 'регистрация', 'аккаунт', 
        'сайт', 'сервис', 'веб', 'мобильный', 'аппка', 'ui', 'ux',
        'лаг', 'зависает', 'краш', 'ошибка', 'баг', 'тормозит',
        'производительность', 'оптимизация', 'нагрузка', 'падение', 
        'стабильность', 'timeout', 'api', 'endpoint', 'backend', 
        'frontend', 'база данных', 'сервер', 'запрос',
        'безопасность', 'авторизация', 'аутентификация', 'пароль', 'доступ'
    ],
    'атмосфера / пространство': [
        'чисто', 'грязно', 'уютно', 'неуютно', 'красиво',
        'дизайн', 'освещение', 'светло', 'тесно', 'пространство',
        'мусор', 'запах', 'шум', 'интерьер', 'комфорт'
    ]
}

missions = {
    'кофе / перекус': [
        'кофе', 'капучино', 'латте', 'эспрессо', 'американо',
        'круассан', 'булочка', 'выпечка', 'перекус', 'перекусить',
        'сироп', 'десерт', 'печенье', 'сэндвич', 'тарт'
    ],
    'готовая еда': [
        'салат', 'суп', 'обед', 'ужин', 'второе', 'гарнир',
        'готовый', 'кулинария', 'контейнер', 'разогреть',
        'паста', 'пицца', 'котлета', 'рис', 'лапша'
    ],
    '“забежать быстро”': [
        'забежать', 'зайти', 'быстро', 'на минуту', 'по пути',
        'мимоходом', 'в спешке', 'взял и ушел', 'вода', 'бутылка',
        'жвачка', 'сигареты', 'мелочь', 'очень быстро'
    ],
    'использование сервиса / приложения': [
        'приложение', 'интерфейс', 'логин', 'регистрация', 'аккаунт',
        'сайт', 'сервис', 'веб', 'мобильный', 'аппка', 'ui', 'ux',
        'ошибка', 'пароль', 'код', 'qr', 'зависает', 'зайти'
    ],
    'базовые покупки': [
        'хлеб', 'молоко', 'яйца', 'масло', 'сахар', 'соль',
        'продукты', 'список', 'закупка', 'домой', 'овощи',
        'фрукты', 'макароны', 'крупа', 'базовые'
    ]
}

df['Ключевая миссия'] = 'базовые покупки' 
df['Ключевая тема'] = 'атмосфера / пространство'

for index, row in df.iterrows():
    text = row['clean_text']
    for category, keywords in  themes.items():
        if any(word in text for word in keywords):
            df.at[index, 'Ключевая тема'] = category
            break

for index, row in df.iterrows():
    text = row['clean_text']
    for category, keywords in  missions.items():
        if any(word in text for word in keywords):
            df.at[index, 'Ключевая миссия'] = category
            break

In [8]:
df

,№,Сеть / формат,Адрес точки,Источник,Текст отзыва,Оценка,Дата,Город,clean_text,Ключевая миссия,Ключевая тема
0,1,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,"Удобный магазин, вкусный кофе и выпечка. Быстр...",5.0,2025-10-18,Москва,удобный магазин вкусный кофе и выпечка быс...,кофе / перекус,ассортимент
1,2,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,"Доставили сюда посылку 5post, было указано что...",2.0,2026-04-27,Москва,доставить сюда посылка 5post было указать чт...,базовые покупки,скорость
2,3,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,"Расположение удобное, продавцы улыбчивые, прия...",3.0,2025-11-27,Москва,расположение удобный продавец улыбчивый пр...,кофе / перекус,качество еды
3,4,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,"Нормальный магазин. Не очень большой выбор, но...",4.0,2026-01-02,Москва,нормальный магазин не очень большой выбор ...,базовые покупки,ассортимент
4,5,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,Кофе 3 в 1 80₽ и говорят такая цена. Через мес...,1.0,2025-11-30,Москва,кофе 3 в 1 80 и говорить такой цена через ...,кофе / перекус,цена
...,...,...,...,...,...,...,...,...,...,...,...
252,253,Налету,NaN,Habr,"Кажется, сегодня ту Пятерочку настиг хабраэффе...",NaN,2021-02-24,NaN,казаться сегодня тот пятёрочка настигнуть ха...,базовые покупки,атмосфера / пространство
253,254,Налету,NaN,Habr,"Где же мои глупые дети работать то будут, на п...",NaN,2021-03-01,NaN,где же мой глупый ребёнок работать то быть н...,базовые покупки,атмосфера / пространство
254,255,Налету,NaN,Habr,"Ну, что заголовок кликбэйтный, это понятно :)\...",NaN,2021-03-08,NaN,ну что заголовок кликбэйтный это понятный ...,базовые покупки,цена
255,256,Налету,NaN,Habr,Пользуюсь сервисом в одном из Зеленоградских П...,NaN,2021-05-05,NaN,пользоваться сервис в одном из зеленоградский ...,“забежать быстро”,ассортимент


In [9]:
review_list = df['Текст отзыва'].astype(str).tolist()
sent_class = sentiment_pipeline(review_list, truncation=True, max_length=512)

df['Сентимент'] = [class_['label'] for class_ in sent_class]

In [10]:
df['Сентимент'].value_counts()

Сентимент
neutral     122
positive    107
negative     28
Name: count, dtype: int64

In [11]:
def sentiment_correction(row):
    sentiment = row['Сентимент']
    raiting = row['Оценка']

    if pd.isna(raiting):
        return sentiment
    
    if raiting == 5.0 and sentiment in ['neutral', 'negative']:
        return 'positive'
        
    elif raiting == 4.0 and sentiment == 'negative':
        return 'neutral'
        
    elif raiting in [1.0, 2.0] and sentiment in ['neutral', 'positive']:
        return 'negative'

    return sentiment

df['Сентимент'] = df.apply(sentiment_correction, axis=1)

In [12]:
df['Сентимент'].value_counts()

Сентимент
positive    119
neutral      92
negative     46
Name: count, dtype: int64

In [13]:
translate = {'positive': 'Позитивный', 'neutral': 'Нейтральный', 'negative': 'Негативный'}

def trasnlation(text):
    '''
    Перевод
    '''
    text = str(text).lower()

    for eng, rus in translate.items():
        text = re.sub(rf'\b{eng}\b', rus, text)

    return text

df['Сентимент'] = df['Сентимент'].apply(trasnlation)

In [14]:
df

,№,Сеть / формат,Адрес точки,Источник,Текст отзыва,Оценка,Дата,Город,clean_text,Ключевая миссия,Ключевая тема,Сентимент
0,1,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,"Удобный магазин, вкусный кофе и выпечка. Быстр...",5.0,2025-10-18,Москва,удобный магазин вкусный кофе и выпечка быс...,кофе / перекус,ассортимент,Позитивный
1,2,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,"Доставили сюда посылку 5post, было указано что...",2.0,2026-04-27,Москва,доставить сюда посылка 5post было указать чт...,базовые покупки,скорость,Негативный
2,3,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,"Расположение удобное, продавцы улыбчивые, прия...",3.0,2025-11-27,Москва,расположение удобный продавец улыбчивый пр...,кофе / перекус,качество еды,Нейтральный
3,4,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,"Нормальный магазин. Не очень большой выбор, но...",4.0,2026-01-02,Москва,нормальный магазин не очень большой выбор ...,базовые покупки,ассортимент,Позитивный
4,5,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,Кофе 3 в 1 80₽ и говорят такая цена. Через мес...,1.0,2025-11-30,Москва,кофе 3 в 1 80 и говорить такой цена через ...,кофе / перекус,цена,Негативный
...,...,...,...,...,...,...,...,...,...,...,...,...
252,253,Налету,NaN,Habr,"Кажется, сегодня ту Пятерочку настиг хабраэффе...",NaN,2021-02-24,NaN,казаться сегодня тот пятёрочка настигнуть ха...,базовые покупки,атмосфера / пространство,Нейтральный
253,254,Налету,NaN,Habr,"Где же мои глупые дети работать то будут, на п...",NaN,2021-03-01,NaN,где же мой глупый ребёнок работать то быть н...,базовые покупки,атмосфера / пространство,Негативный
254,255,Налету,NaN,Habr,"Ну, что заголовок кликбэйтный, это понятно :)\...",NaN,2021-03-08,NaN,ну что заголовок кликбэйтный это понятный ...,базовые покупки,цена,Нейтральный
255,256,Налету,NaN,Habr,Пользуюсь сервисом в одном из Зеленоградских П...,NaN,2021-05-05,NaN,пользоваться сервис в одном из зеленоградский ...,“забежать быстро”,ассортимент,Нейтральный


In [15]:
def_wow = [
    'удивить', 'фаворит', 'безупречный', 'любить', 
    'впечатлить', 'топ', 'восторг', 'советовать', 'идеальный',
    'отличный', 'крутой', 'любимый', 'превосходный',
    'вау', 'потрясать', 'рекомендовать', 'обожать', 'замечательный',
    'великолепный', 'превзойти', 'шикарный', 'супер']

def_neg = [
    'дно', 'ужасный', 'просрочка', 'отвратительный', 'отравиться',
    'испортить', 'нахамить', 'грязный', 'плесень', 'грубость',
    'просроченный', 'возврат', 'вонять', 'тухлый', 'отравление',
    'грязь', 'хамство', 'кошмар', 'насекомое', 'обман',
    'обмануть', 'списать', 'антисанитария', 'мусор', 'таракан',
    'грубый', 'несвежий']

tech_wow = ['интуитивный', 'инновация', 'технологичный']

tech_neg = [
    'сломать', 'лагает', 'утечка', 'краш', 'лаг',
    'зависать', 'слив', 'таймаут', 'ошибка', 'тормозить',
    'сбой', 'timeout', 'костыль', 'взлом', 'глючит',
    'фризит', 'уязвимость', 'вылетать', 'баг']

df['Сильный позитив'] = 0
df['Критичный негатив'] = 0

for index, row in df.iterrows():
    text = str(row['clean_text']) 

    sentiment = str(row['Сентимент']).lower()
    rating = row['Оценка'] 

    with_wow_words = any(word in text for word in def_wow) or any(word in text for word in tech_wow)

    if with_wow_words and ('позитив' in sentiment):
        if pd.isna(rating) or rating == 5.0:
            df.at[index, 'Сильный позитив'] = 1

    with_neg_words = any(word in text for word in def_neg) or any(word in text for word in tech_neg)

    if with_neg_words and ('негатив' in sentiment):
        if pd.isna(rating) or rating == 1.0:
            df.at[index, 'Критичный негатив'] = 1

In [16]:
(df['Сильный позитив']).sum()

np.int64(37)

In [17]:
(df['Критичный негатив']).sum()

np.int64(14)

In [18]:
df

,№,Сеть / формат,Адрес точки,Источник,Текст отзыва,Оценка,Дата,Город,clean_text,Ключевая миссия,Ключевая тема,Сентимент,Сильный позитив,Критичный негатив
0,1,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,"Удобный магазин, вкусный кофе и выпечка. Быстр...",5.0,2025-10-18,Москва,удобный магазин вкусный кофе и выпечка быс...,кофе / перекус,ассортимент,Позитивный,0,0
1,2,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,"Доставили сюда посылку 5post, было указано что...",2.0,2026-04-27,Москва,доставить сюда посылка 5post было указать чт...,базовые покупки,скорость,Негативный,0,0
2,3,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,"Расположение удобное, продавцы улыбчивые, прия...",3.0,2025-11-27,Москва,расположение удобный продавец улыбчивый пр...,кофе / перекус,качество еды,Нейтральный,0,0
3,4,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,"Нормальный магазин. Не очень большой выбор, но...",4.0,2026-01-02,Москва,нормальный магазин не очень большой выбор ...,базовые покупки,ассортимент,Позитивный,0,0
4,5,Налету,"Москва, ул. Барклая, 6А, корп. 1",Yandex,Кофе 3 в 1 80₽ и говорят такая цена. Через мес...,1.0,2025-11-30,Москва,кофе 3 в 1 80 и говорить такой цена через ...,кофе / перекус,цена,Негативный,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
252,253,Налету,NaN,Habr,"Кажется, сегодня ту Пятерочку настиг хабраэффе...",NaN,2021-02-24,NaN,казаться сегодня тот пятёрочка настигнуть ха...,базовые покупки,атмосфера / пространство,Нейтральный,0,0
253,254,Налету,NaN,Habr,"Где же мои глупые дети работать то будут, на п...",NaN,2021-03-01,NaN,где же мой глупый ребёнок работать то быть н...,базовые покупки,атмосфера / пространство,Негативный,0,0
254,255,Налету,NaN,Habr,"Ну, что заголовок кликбэйтный, это понятно :)\...",NaN,2021-03-08,NaN,ну что заголовок кликбэйтный это понятный ...,базовые покупки,цена,Нейтральный,0,0
255,256,Налету,NaN,Habr,Пользуюсь сервисом в одном из Зеленоградских П...,NaN,2021-05-05,NaN,пользоваться сервис в одном из зеленоградский ...,“забежать быстро”,ассортимент,Нейтральный,0,0


In [19]:
df.to_excel('reviews_visualisation_ready.xlsx')